# PyADM1ODE: Parallel Simulation Example

Run multiple ADM1 scenarios concurrently to compare feed rates and sweep model parameters.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/PyADM1ODE/blob/master/examples/colab_05_parallel_two_stage_simulation.ipynb)

## 1. Setup

In [ ]:
!git clone https://github.com/dgaida/PyADM1ODE.git
%cd PyADM1ODE
!pip install -e .

## 2. Imports

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import pyadm1
from pyadm1 import Feedstock
from pyadm1.core.adm1 import ADM1, get_state_zero_from_csv
from pyadm1.simulation.parallel import (
    ParallelSimulator,
    ParameterSweepConfig,
)

REPO_ROOT = Path(pyadm1.__file__).resolve().parents[1]

## 3. Build the base ADM1 model

In [ ]:
duration = 30.0 # days
feedstock = Feedstock(
    ["maize_silage_milk_ripeness", "swine_manure"],
    feeding_freq=24,
    total_simtime=int(duration) + 1,
)
base_feed = [11.4, 6.1, 0, 0, 0, 0, 0, 0, 0, 0] # m^3/d: maize, swine

# Digester
adm1 = ADM1(feedstock, V_liq=1200.0, V_gas=216.0, T_ad=315.15)
adm1.set_influent_dataframe(feedstock.get_influent_dataframe(Q=base_feed))
adm1.create_influent(base_feed, 0)

# Steady-state initial state
initial_state = get_state_zero_from_csv(
    str(REPO_ROOT / "data" / "initial_states" / "digester_initial8.csv")
)

# Parallel runner with 4 workers
parallel = ParallelSimulator(adm1, n_workers=4, verbose=False)

## 4. Run feed-rate scenarios in parallel

In [ ]:
feed_scenarios = [
    ("Low",       [ 8.0,  4.0, 0, 0, 0, 0, 0, 0, 0, 0]),
    ("Base",      [11.4,  6.1, 0, 0, 0, 0, 0, 0, 0, 0]),
    ("High",      [15.0,  8.0, 0, 0, 0, 0, 0, 0, 0, 0]),
    ("Very High", [20.0, 10.0, 0, 0, 0, 0, 0, 0, 0, 0]),
]

scenarios = [{"Q": Q} for _, Q in feed_scenarios]

results = parallel.run_scenarios(
    scenarios=scenarios,
    duration=duration,
    initial_state=initial_state,
    dt=1.0,
    compute_metrics=True,
)

## 5. Compare results

In [ ]:
print(f"{'Scenario':<12s} {'Q_total':>8s} {'Q_gas':>10s} {'Q_ch4':>10s} {'pH':>6s}")
print("-" * 52)
for (label, Q), r in zip(feed_scenarios, results):
    m = r.metrics
    print(
        f"{label:<12s} {sum(Q):>8.1f} {m.get('Q_gas', 0):>10.1f} "
        f"{m.get('Q_ch4', 0):>10.1f} {m.get('pH', 0):>6.2f}"
    )

## 6. Plot methane production

In [ ]:
labels = [label for label, _ in feed_scenarios]
q_ch4 = [r.metrics.get("Q_ch4", 0) for r in results]

plt.figure(figsize=(8, 5))
plt.bar(labels, q_ch4, color="steelblue")
plt.ylabel("Methane production [m³/d]")
plt.title("Methane yield by feed rate")
plt.grid(axis="y", alpha=0.3)
plt.show()

## 7. Parameter sweep: acetate uptake rate `k_m_ac`

The default is 8 d⁻¹ (at 35 °C). Lower values throttle methanogenesis from acetate; higher values relieve acetate accumulation.

In [ ]:
sweep_config = ParameterSweepConfig(
    parameter_name="k_m_ac",
    values=[2.0, 3.0, 3.5, 5.0, 8.0],
    other_params={"Q": base_feed},
)

sweep_results = parallel.parameter_sweep(
    config=sweep_config,
    duration=duration,
    initial_state=initial_state,
)

In [ ]:
k_values = sweep_config.values
ch4 = [r.metrics.get("Q_ch4", 0) for r in sweep_results]

best = int(np.argmax(ch4))
print(f"Best k_m_ac = {k_values[best]:.2f} -> CH4 = {ch4[best]:.1f} m³/d")

plt.figure(figsize=(8, 5))
plt.plot(k_values, ch4, marker="o")
plt.xlabel("k_m_ac [1/d]")
plt.ylabel("Methane production [m³/d]")
plt.title("Methane vs. acetate uptake rate")
plt.grid(True, alpha=0.3)
plt.show()